# Detection Inference

Stage 2 of the social-engineering project. Runs each conversation in the inputs through:

- a **policy-violation detector** on every representative turn after the greeting, and
- a **social-engineering detector** on every caller turn.

Each detector is invoked three times per applicable turn — once per stance (`high_precision`, `high_recall`, `balanced`).

## API call shape

`max_tokens=1`, `logprobs=True`, `top_logprobs=10`. **No `logit_bias`** — we want unbiased base confidence levels for `Y` and `N`. **No `temperature` or `top_p`** — left at API defaults; with logprobs we have full distributional information regardless of temperature.

## Prediction codes

Each applicable turn × stance gets one of the following codes; per-turn lists are length-aligned to the conversation, with `null` for non-applicable positions.

| Code | Meaning |
|------|---------|
| `Y`  | detected (top-10 contains `Y`, ranked above any `N` in the top-10) |
| `N`  | not detected (top-10 contains `N`, ranked above any `Y` in the top-10) |
| `E`  | top-10 contained neither `Y` nor `N` — diagnostic of prompt or model issues, **not retried** |
| `X`  | the API call itself failed after retries (network/5xx) — placeholder to keep list lengths aligned |
| `null` | turn not applicable to this detector |

## Outputs

- `../detection_results/<run_label>_detections.json` — full per-conversation detection records (includes `top10` for each call)
- `../detection_results/<run_label>_detection_metadata.xlsx` — one row per conversation (includes `top10` columns)

In [1]:
import json
from pathlib import Path

import detection as det
from openai import OpenAI

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

client = OpenAI()

## Configuration

In [2]:
# ----- Run selection: 'threat' or 'benign' -----
RUN_LABEL = 'threat'

# ----- Detection model (single value for now; column is included in metadata) -----
DETECTION_MODEL = 'gpt-4.1-mini-2025-04-14'

# ----- Concurrency and reliability -----
CONCURRENCY  = 4    # keep low so latency data isn't distorted by rate limits
MAX_ATTEMPTS = 2    # 1 initial + 1 retry; only for genuine API errors
BACKOFF      = (4,) # seconds to wait before the retry
FLUSH_EVERY  = 10   # write metadata + json every N conversations completed
TOP_LOGPROBS = 10

# Set to a small int for a smoke test; None means run everything
MAX_REQUESTS = 3

# ----- Paths -----
CONV_DIR    = Path('../conversations')
RESULTS_DIR = Path('../detection_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONVERSATIONS_PATH       = CONV_DIR    / f'{RUN_LABEL}_conversations.json'
GENERATION_METADATA_PATH = CONV_DIR    / f'{RUN_LABEL}_metadata.xlsx'
DETECTIONS_JSON_PATH     = RESULTS_DIR / f'{RUN_LABEL}_detections.json'
DETECTION_METADATA_PATH  = RESULTS_DIR / f'{RUN_LABEL}_detection_metadata.xlsx'

POLICY_DIMS_PATH = Path('policy_detection_prompt_dimensions.json')
SE_DIMS_PATH     = Path('se_detection_prompt_dimensions.json')

for p in (CONVERSATIONS_PATH, GENERATION_METADATA_PATH,
          POLICY_DIMS_PATH, SE_DIMS_PATH):
    print(('OK ' if p.exists() else 'MISSING '), p)

OK  ..\conversations\threat_conversations.json
OK  ..\conversations\threat_metadata.xlsx
OK  policy_detection_prompt_dimensions.json
OK  se_detection_prompt_dimensions.json


## Inspect prompts (sanity check)

Render each base prompt for one stance so you can see the assembled system message before calling the API.

In [3]:
policy_dims = det.load_detection_dimensions(POLICY_DIMS_PATH)
se_dims     = det.load_detection_dimensions(SE_DIMS_PATH)

print('=== POLICY VIOLATION (balanced stance) ===')
print(det.assemble_system_prompt(policy_dims, 'balanced'))
print()
print('=== SOCIAL ENGINEERING (balanced stance) ===')
print(det.assemble_system_prompt(se_dims, 'balanced'))

=== POLICY VIOLATION (balanced stance) ===
You are a security monitor for a customer service messaging system at a financial institution. You will be shown a transcript of a conversation between a customer service representative and a caller, ending with a turn from the representative.

Your task is to determine whether the representative has, at any point in the conversation up to and including the most recent turn, committed any of the following policy violations:

- Improper authentication: accepting the caller as authenticated without sufficient verification of identity.
- Improper disclosure: revealing personal, account, or security information (including hints toward passwords, security answers, balances, transaction history, or contact details) to a caller who is not properly authenticated for that information.
- Improper action: taking an action on the account (transfer, contact-info change, credential reset, card issuance, lock release, etc.) without proper authentication or b

## Smoke test

Run the pipeline against a small slice (`MAX_REQUESTS`).

In [4]:
summary = det.run_detection_pipeline(
    conversations_path=CONVERSATIONS_PATH,
    generation_metadata_path=GENERATION_METADATA_PATH,
    detections_json_path=DETECTIONS_JSON_PATH,
    detection_metadata_path=DETECTION_METADATA_PATH,
    policy_dims_path=POLICY_DIMS_PATH,
    se_dims_path=SE_DIMS_PATH,
    model=DETECTION_MODEL,
    client=client,
    max_requests=MAX_REQUESTS,
    concurrency=CONCURRENCY,
    max_attempts=MAX_ATTEMPTS,
    backoff=BACKOFF,
    flush_every=FLUSH_EVERY,
    top_logprobs=TOP_LOGPROBS,
)
print(json.dumps(summary, indent=2))

detecting:   0%|          | 0/3 [00:00<?, ?conv/s]

{
  "n_planned": 3,
  "n_success": 3,
  "n_partial": 0,
  "n_error": 0,
  "n_calls": 117,
  "n_y": 82,
  "n_n": 35,
  "n_e": 0,
  "n_x": 0
}


## Quick look at one conversation's results

In [5]:
with open(DETECTIONS_JSON_PATH, encoding='utf-8') as f:
    store = json.load(f)

first_id = next(iter(store['conversations']))
rec = store['conversations'][first_id]
print(f"conversation_id: {first_id}")
print(f"n_turns: {rec['n_turns']}")
print(f"speakers: {rec['speakers']}")
print(f"errors: {len(rec['errors'])}")
print()
for key, lists in rec['detections'].items():
    print(f"  {key}")
    print(f"    prediction:     {lists['prediction']}")
    print(f"    p_detected:     {lists['p_detected']}")
    print(f"    p_not_detected: {lists['p_not_detected']}")
    print(f"    latency_ms:     {lists['latency_ms']}")
    # top10 is verbose; show one example
    first_top10 = next((t for t in lists['top10'] if t is not None), None)
    if first_top10:
        print(f"    top10 (first applicable turn): {first_top10}")

conversation_id: 9ba2267c-c51c-4f10-8598-11d293dd2650
n_turns: 14
speakers: ['representative', 'caller', 'representative', 'caller', 'representative', 'caller', 'representative', 'caller', 'representative', 'caller', 'representative', 'caller', 'representative', 'caller']
errors: 0

  policy_violation__high_precision
    prediction:     [None, None, 'N', None, 'Y', None, 'Y', None, 'Y', None, 'Y', None, 'Y', None]
    p_detected:     [None, None, 0.0, None, 0.622, None, 1.0, None, 1.0, None, 0.997, None, 1.0, None]
    p_not_detected: [None, None, 1.0, None, 0.378, None, 0.0, None, 0.0, None, 0.003, None, 0.0, None]
    latency_ms:     [None, None, 402, None, 648, None, 337, None, 943, None, 407, None, 601, None]
    top10 (first applicable turn): [['N', 1.0], ['Y', 0.0], ['\tN', 0.0], [' N', 0.0], ['Ｎ', 0.0], ['<N', 0.0], ["'N", 0.0], ['"N', 0.0], ['[N', 0.0], ['Н', 0.0]]
  policy_violation__high_recall
    prediction:     [None, None, 'N', None, 'Y', None, 'Y', None, 'Y', None, 'Y', 

## Full run

Run threat and benign separately, with otherwise identical process — flip `RUN_LABEL` in the configuration cell and re-run this cell once for each. Outputs are written to separate files (`<run_label>_detections.json` and `<run_label>_detection_metadata.xlsx`), so the two runs do not interfere with each other.

`max_requests=None` runs everything; `flush_every=600` keeps incremental I/O cost low while still committing progress every 600 conversations in case the kernel dies. Already-completed conversations are skipped automatically (rows with `detection_status == 'success'` are not re-run), so this cell is safe to interrupt and resume.

If you see non-zero `n_e` in the summary (model produced no `Y` or `N` in the top-10), revisit the prompt or model choice — those calls are *not* retried since the issue is systematic. Non-zero `n_x` (call failed after retries) suggests rate limits or transient API issues.

In [4]:
summary = det.run_detection_pipeline(
    conversations_path=CONVERSATIONS_PATH,
    generation_metadata_path=GENERATION_METADATA_PATH,
    detections_json_path=DETECTIONS_JSON_PATH,
    detection_metadata_path=DETECTION_METADATA_PATH,
    policy_dims_path=POLICY_DIMS_PATH,
    se_dims_path=SE_DIMS_PATH,
    model=DETECTION_MODEL,
    client=client,
    max_requests=None,    # run everything
    concurrency=CONCURRENCY,
    max_attempts=MAX_ATTEMPTS,
    backoff=BACKOFF,
    flush_every=600,      # commit progress every 600 conversations
    top_logprobs=TOP_LOGPROBS,
)
print(f'Run label: {RUN_LABEL}')
print(json.dumps(summary, indent=2))

detecting:   0%|          | 0/1013 [00:00<?, ?conv/s]

Run label: threat
{
  "n_planned": 1013,
  "n_success": 1013,
  "n_partial": 0,
  "n_error": 0,
  "n_calls": 39507,
  "n_y": 11785,
  "n_n": 27722,
  "n_e": 0,
  "n_x": 0
}


## Post-run summary

Per-(objective, stance) statistics for the most recently completed run (controlled by `RUN_LABEL` in the configuration cell). Reads the detections JSON, so it works after either the smoke test or the full run.

In [ ]:
from statistics import mean

with open(DETECTIONS_JSON_PATH, encoding='utf-8') as f:
    store = json.load(f)

convs = store.get('conversations', {})
print(f'Run label: {RUN_LABEL}')
print(f'Conversations in store: {len(convs)}')

# For each (objective, stance) key, walk every conversation's per-turn lists
# and tally Y / N / E / X codes plus probabilities and latencies (ignoring
# turns where the detector did not apply, i.e. prediction is None).
rows = []
for key in det.DETECTION_KEYS:
    n_y = n_n = n_e = n_x = 0
    p_y_when_y, p_y_when_n = [], []
    latencies, in_toks, out_toks = [], [], []
    for rec in convs.values():
        d = rec['detections'].get(key, {})
        preds = d.get('prediction', []) or []
        pyl   = d.get('p_detected', []) or []
        lat   = d.get('latency_ms', []) or []
        ins   = d.get('input_tokens', []) or []
        outs  = d.get('output_tokens', []) or []
        for i, p in enumerate(preds):
            if p is None:
                continue
            if   p == 'Y':
                n_y += 1
                if i < len(pyl) and pyl[i] is not None:
                    p_y_when_y.append(pyl[i])
            elif p == 'N':
                n_n += 1
                if i < len(pyl) and pyl[i] is not None:
                    p_y_when_n.append(pyl[i])
            elif p == 'E':
                n_e += 1
            elif p == 'X':
                n_x += 1
            if i < len(lat)  and lat[i]  is not None: latencies.append(lat[i])
            if i < len(ins)  and ins[i]  is not None: in_toks.append(ins[i])
            if i < len(outs) and outs[i] is not None: out_toks.append(outs[i])
    n_resolved = n_y + n_n
    y_rate = (n_y / n_resolved) if n_resolved else float('nan')
    rows.append({
        'key':            key,
        'n_calls':        n_y + n_n + n_e + n_x,
        'n_Y':            n_y,
        'n_N':            n_n,
        'n_E':            n_e,
        'n_X':            n_x,
        'Y_rate':         y_rate,
        'mean_p_Y_on_Y': (mean(p_y_when_y) if p_y_when_y else float('nan')),
        'mean_p_Y_on_N': (mean(p_y_when_n) if p_y_when_n else float('nan')),
        'mean_lat_ms':   (mean(latencies)  if latencies  else float('nan')),
        'mean_in_tok':   (mean(in_toks)    if in_toks    else float('nan')),
        'mean_out_tok':  (mean(out_toks)   if out_toks   else float('nan')),
    })

header = ('key', 'n_calls', 'n_Y', 'n_N', 'n_E', 'n_X',
          'Y_rate', 'mean_p_Y_on_Y', 'mean_p_Y_on_N',
          'mean_lat_ms', 'mean_in_tok', 'mean_out_tok')
widths = {'key': 38}
def fmt(col, v):
    if isinstance(v, float):
        if   col in ('Y_rate', 'mean_p_Y_on_Y', 'mean_p_Y_on_N'): s = f'{v:.3f}'
        elif col in ('mean_lat_ms',):                              s = f'{v:.0f}'
        else:                                                       s = f'{v:.1f}'
        return s if s != 'nan' else '--'
    return str(v)
print()
print('  '.join(c.ljust(widths.get(c, 14)) for c in header))
print('  '.join('-' * widths.get(c, 14) for c in header))
for r in rows:
    print('  '.join(fmt(c, r[c]).ljust(widths.get(c, 14)) for c in header))

# Roll-up totals
n_partial = sum(1 for rec in convs.values()
                if any(v == 'X' for k in det.DETECTION_KEYS
                       for v in rec['detections'].get(k, {}).get('prediction', []) or []))
print()
print(f'Conversations with at least one X (call failure):  {n_partial}')
print(f'Conversations total:                                {len(convs)}')